# SolarScan AI — Custom YOLOv8 Defect Detection Training Pipeline

This Google Colab notebook trains a custom **YOLOv8** object detection model on real-world solar panel defect datasets (Hotspots, Micro-cracks, and Soiling), and exports the trained weights to a mobile-friendly **INT8 Quantized TFLite** format.

### Instructions to Run:
1. Open this notebook in **Google Colab** (https://colab.research.google.com).
2. Enable GPU acceleration: Go to **Runtime** -> **Change runtime type** -> select **GPU (T4 GPU)**.
3. Click **Runtime** -> **Run all** to start the pipeline.

### Step 1: Install Dependencies
We install the Ultralytics YOLOv8 library and other utilities.

In [ ]:
!pip install ultralytics roboflow pillow opencv-python
import os
from ultralytics import YOLO
print("Setup complete. YOLOv8 ready.")

### Step 2: Download Your Datasets
You can pull datasets from Roboflow using their API wrapper, or upload zip files manually to the `/content/` directory.

Replace the mock API credentials below with your actual Roboflow project details when ready.

In [ ]:
# Download dataset from Roboflow
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")  # Replace with actual API key
project = rf.workspace("roboflow-100").project("solar-panels-taxvb")
version = project.version(1)
dataset = version.download("yolov8")

print("Dataset downloaded to:", dataset.location)

### Step 3: Configure `dataset.yaml`
YOLOv8 requires a config mapping your directories to classes. We create a template file listing the 9 defect classes from your React application.

In [ ]:
yaml_content = """
path: /content/solar-panels-1/     # Root dataset directory
train: train/images
val: valid/images
test: test/images

names:
  0: hotspot
  1: crack
  2: soiling
  3: bypass_failure
  4: delamination
  5: discoloration
  6: snail_trail
  7: pid
  8: snow_cover
"""

with open('/content/dataset.yaml', 'w') as f:
    f.write(yaml_content.strip())

print("dataset.yaml config written successfully!")

### Step 4: Run YOLOv8 Model Training
We load a pre-trained YOLOv8 nano model weights (`yolov8n.pt`) and fine-tune it on the clean energy data classes for 100 epochs.

In [ ]:
# Load pretrained model
model = YOLO('yolov8n.pt')

# Start training
results = model.train(
    data='/content/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0, # Use GPU
    project='solar_scans_project',
    name='solar_defect_yolov8'
)
print("Training cycle complete!")

### Step 5: Evaluate & Validate the Model
Run validation tests on the test set to evaluate mean Average Precision (mAP) scores.

In [ ]:
metrics = model.val()
print("Validation mAP@50 score:", metrics.box.map50)

### Step 6: Export to Quantized TFLite (INT8) Format
We export the PyTorch model (`.pt`) to TensorFlow Lite format with INT8 quantization so it runs at maximum speed on mobile device hardware.

In [ ]:
# Export trained model to TFLite format
tflite_path = model.export(format='tflite', int8=True)
print("PWA Quantized TFLite weights saved at:", tflite_path)

### Step 7: How to Download & Link Weights to your API
1. Open the file explorer sidebar on the left side of your Google Colab window.
2. Locate the file: `/content/solar_scans_project/solar_defect_yolov8/weights/best.pt` (PyTorch model) or `/content/solar_scans_project/solar_defect_yolov8/weights/best_saved_model/best_int8.tflite`.
3. Right-click and choose **Download**.
4. Place the downloaded **`best.pt`** file inside the `backend/` folder of your SolarScan AI application to run real detections.